# Formula Explorer

Type a formula, get a slider for every constant in it, and watch the graph move as you drag.

A formula like $A\sin(\omega t + \phi)$ is not one curve --- it is a whole **family**, one curve for each choice of $A$, $\omega$, $\phi$. Those constants are the **parameters**; the letter on the horizontal axis is the **input**. This notebook lets you turn the parameters and see what each one does.

**To use it:** Cell &rarr; Run All, then work in the app in section 2.

## 1. Setup

Run this cell once. It defines the app --- nothing in it needs reading or editing.

In [ ]:
# --- Run this cell once. It sets up the app; nothing here needs reading. ---
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.5, 4.6)


def center_axes(ax, xlabel='x', ylabel='y'):
    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel(xlabel, loc='right')
    ax.set_ylabel(ylabel, loc='top', rotation=0)
    return ax


def parse_formula(text, variable=None):
    expr = sp.sympify(text)
    symbols = sorted(expr.free_symbols, key=lambda s: s.name)
    if variable is None:
        names = [s.name for s in symbols]
        variable = next((c for c in ('x', 't') if c in names), names[0] if names else 'x')
    var = sp.Symbol(variable)
    return expr, var, [s for s in symbols if s != var]


def evaluate(expr, var, params, values, xs):
    f = sp.lambdify((var,) + tuple(params), expr, modules=['numpy'])
    with np.errstate(all='ignore'):
        ys = f(xs, *[values[p.name] for p in params])
    ys = np.asarray(ys, dtype=float)
    if ys.shape != xs.shape:                       # a constant formula returns one number
        ys = np.full_like(xs, float(ys))
    ys[~np.isfinite(ys)] = np.nan                  # undefined points leave a gap
    return ys


def break_at_jumps(ys, factor=50.0):
    ys = ys.copy()
    steps = np.abs(np.diff(ys))
    typical = np.nanmedian(steps)
    if np.isfinite(typical) and typical > 0:
        ys[1:][steps > factor * typical] = np.nan  # cut the line at an asymptote
    return ys


def nice_ylim(ys, pad=0.15):
    finite = ys[np.isfinite(ys)]
    if finite.size == 0:
        return (-1.0, 1.0)
    lo, hi = np.percentile(finite, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        lo, hi = float(np.min(finite)) - 1.0, float(np.max(finite)) + 1.0
    span = hi - lo
    return (lo - pad * span, hi + pad * span)


def formula_explorer(formula='A*sin(w*t + phi)', variable='t',
                     slider_range=(-5.0, 5.0), step=0.1, xlim=(-6.5, 6.5)):
    wide = widgets.Layout(width='420px')
    narrow = widgets.Layout(width='190px')
    label_style = {'description_width': '70px'}

    formula_box = widgets.Text(value=formula, description='f =', layout=wide,
                               style=label_style, continuous_update=False)
    var_box = widgets.Text(value=variable, description='variable', layout=narrow,
                           style=label_style, continuous_update=False)
    xmin_box = widgets.FloatText(value=xlim[0], description='x min', layout=narrow,
                                 style=label_style)
    xmax_box = widgets.FloatText(value=xlim[1], description='x max', layout=narrow,
                                 style=label_style)
    pin_button = widgets.Button(description='Pin current curve')
    clear_button = widgets.Button(description='Clear pins')
    message = widgets.HTML()
    slider_box = widgets.VBox([])
    plot_out = widgets.Output()

    state = {'sliders': {}, 'pins': [], 'parsed': None}

    def draw(*_):
        parsed = state['parsed']
        with plot_out:
            plot_out.clear_output(wait=True)
            if parsed is None:
                return
            expr, var, params = parsed
            values = {name: s.value for name, s in state['sliders'].items()}
            xs = np.linspace(xmin_box.value, xmax_box.value, 1000)
            ys = break_at_jumps(evaluate(expr, var, params, values, xs))

            fig, ax = plt.subplots()
            for pinned_ys, caption in state['pins']:
                ax.plot(xs, pinned_ys, linestyle='--', color='0.6', linewidth=1.2, label=caption)
            ax.plot(xs, ys, color='C0', linewidth=2.0, label='current')
            ax.set_xlim(xs[0], xs[-1])
            ax.set_ylim(*nice_ylim(np.concatenate([ys] + [p for p, _ in state['pins']])))
            center_axes(ax, xlabel=str(var))
            ax.set_title(f'${sp.latex(expr)}$', fontsize=12, pad=16)
            if state['pins']:
                ax.legend(loc='lower right', fontsize=8, frameon=False)
            plt.show()
            plt.close(fig)

    def rebuild(*_):
        state['pins'] = []
        try:
            expr, var, params = parse_formula(formula_box.value, var_box.value.strip() or None)
        except Exception as error:
            state['parsed'] = None
            slider_box.children = ()
            message.value = (f"<span style='color:#b00'>Could not read that formula: "
                             f"{type(error).__name__} &mdash; {error}</span>")
            draw()
            return

        state['parsed'] = (expr, var, params)
        message.value = (f"Input <b>{var}</b>; sliders: "
                         f"<b>{', '.join(p.name for p in params) or 'none'}</b>")

        kept = {name: s.value for name, s in state['sliders'].items()}
        state['sliders'] = {}
        for p in params:
            slider = widgets.FloatSlider(value=kept.get(p.name, 1.0),
                                         min=slider_range[0], max=slider_range[1], step=step,
                                         description=p.name, continuous_update=True,
                                         readout_format='.2f', layout=wide, style=label_style)
            slider.observe(draw, names='value')
            state['sliders'][p.name] = slider
        slider_box.children = tuple(state['sliders'].values())
        draw()

    def pin(_):
        if state['parsed'] is None:
            return
        expr, var, params = state['parsed']
        values = {name: s.value for name, s in state['sliders'].items()}
        xs = np.linspace(xmin_box.value, xmax_box.value, 1000)
        caption = ', '.join(f'{k}={v:g}' for k, v in values.items()) or 'pinned'
        state['pins'].append((break_at_jumps(evaluate(expr, var, params, values, xs)), caption))
        draw()

    def clear_pins(_):
        state['pins'] = []
        draw()

    formula_box.observe(rebuild, names='value')
    var_box.observe(rebuild, names='value')
    xmin_box.observe(draw, names='value')
    xmax_box.observe(draw, names='value')
    pin_button.on_click(pin)
    clear_button.on_click(clear_pins)

    display(widgets.VBox([
        widgets.HBox([formula_box, var_box]),
        widgets.HBox([xmin_box, xmax_box, pin_button, clear_button]),
        message,
        slider_box,
        plot_out,
    ]))
    rebuild()

## 2. The Explorer

Run the cell below to open the app. Pick a starting family by editing `choice`, or ignore the dictionary entirely and type your own formula into the box.

**Typing a formula**

| Want | Type | Not |
|---|---|---|
| multiply | `2*x`, `A*sin(w*t)` | `2x`, `A sin(wt)` |
| power | `x**2` | `x^2` |
| root | `sqrt(x)` | `√x` |
| natural log | `log(x)` | `ln(x)` |
| exponential | `exp(k*x)` | `e^(kx)` |

`sin`, `cos`, `tan`, `exp`, `log`, `sqrt`, `Abs`, `pi` and `E` all work. Whatever letter you put in the **variable** box becomes the horizontal axis; every *other* letter gets a slider.

**The controls**

- **f =** the formula --- edit it and press <kbd>Enter</kbd> to rebuild the sliders.
- **variable** which letter is the input, usually `x` or `t`.
- **x min / x max** the window to draw.
- **Pin current curve** freezes a dashed copy so the next drag can be compared against it; **Clear pins** removes them.

In [ ]:
# Formulas to try.  A gap in a curve is a point excluded from the domain, not a
# drawing error -- try 'reciprocal' or 'logarithm' and drag c to watch it move.
FORMULAS = {
    'linear':             ('m*x + b',                  'x'),
    'quadratic':          ('a*x**2 + b*x + c',         'x'),
    'quadratic (vertex)': ('a*(x - h)**2 + k',         'x'),
    'cubic':              ('a*(x-b)*(x-c)*(x-d)',             'x'),
    'exponential':        ('A*exp(k*x)',               'x'),
    'logarithm':          ('a*log(x - c)',             'x'),
    'reciprocal':         ('1/(x - c)',                'x'),
    'even root':          ('sqrt(x - c)',              'x'),
    'sine wave':          ('A*sin(w*t + phi)',         't'),
    'damped wave':        ('A*exp(-d*t)*sin(w*t)',     't'),
    'tangent':            ('tan(w*t)',                 't'),
    'logistic':           ('1/(1 + exp(-k*(x - x0)))', 'x'),
}

choice = 'sine wave'          # <-- any key above

formula, variable = FORMULAS[choice]
formula_explorer(formula, variable=variable)

## 3. Things Worth Trying

- **Shift versus stretch.** In `a*(x - h)**2 + k`, one slider moves the parabola sideways, one moves it up, one changes its width. Predict which is which before you drag.
- **Horizontal moves run against their sign.** Increasing `h` in `(x - h)**2` moves the graph *right*. Pin a curve first, then drag, and watch the two.
- **A gap is a domain restriction, not a bug.** Try `reciprocal` or `logarithm` and drag `c`: the excluded region slides along with it.
- **Some parameters do nothing visible.** In `A*sin(w*t + phi)`, changing `phi` by $2\pi$ lands the wave exactly back where it started.